# Image Captioning with Visual Attention (Flickr8k)

An encoder-decoder captioner where the decoder **attends over a 7×7 grid of image regions** at every generated word.

- **Encoder:** frozen EfficientNetB0 (ImageNet). The 7×7×1280 feature map is kept as 49 region vectors and projected, with learned 2D position embeddings.
- **Decoder:** a 2-layer Transformer decoder. Each layer has causal self-attention over the caption so far, cross-attention over the 49 image regions, and a feed-forward block.
- **Training:** teacher forcing on the full caption at once, with a token-normalised masked cross-entropy (padding contributes nothing). Train/val/test splits are by image, with early stopping on validation loss.
- **Evaluation:** corpus BLEU-1 to BLEU-4 on held-out test images against all 5 reference captions, for greedy and beam search decoding.
- **Interpretability:** per-word cross-attention maps overlaid on the image.

Dataset: [Flickr8k on Kaggle](https://www.kaggle.com/datasets/adityajn105/flickr8k), 8,091 images with 5 captions each. Runs on a Kaggle T4 GPU.

## What changed from the earlier version (and why)

The earlier notebooks (Xception + BiLSTM, and an EfficientNet variant with an "attention" block) had problems that made both the model and the reported BLEU unreliable:

| Issue | Effect | Fix here |
|---|---|---|
| The attention block repeated one image vector `max_length` times (`RepeatVector`) and scored it against every caption state | Every query row was identical, so the context collapsed to a rank-1 rescaling of the summed LSTM states. It never looked at image regions (the image was already pooled to one vector) | Cross-attention from each caption position over 49 spatial regions |
| Attention softmax ignored padding | Padded timesteps received attention weight | Causal masking. With post-padding, real tokens never attend to padding, and padded targets are masked out of the loss |
| `startseq` / `endseq` kept in both references and predictions for BLEU | Inflated BLEU-1/2: `startseq` is always a match | Special tokens are stripped before scoring. BLEU-1 to BLEU-4 are reported |
| `str.replace('[^A-Za-z]', '')` is not a regex | Cleaning was a no-op, so punctuation stayed in tokens | `re.sub` |
| One-letter words were dropped (`"a"`, the most common caption word) | Captions came out ungrammatical | Kept |
| Word2Vec embeddings trained on **all** captions, including test | Test captions leaked into the model | Vocabulary built from training captions only, and embeddings learned end to end |
| `ModelCheckpoint(monitor='val_loss')` with no validation data | The checkpoint never saved a "best" model | Validation split, early stopping and checkpoints on `val_loss` |
| Each caption expanded into one sample per prefix, with one-hot targets | Memory grew with caption length times vocabulary size (the "epochs > 64 → memory error" note) | One sample per caption with integer targets |
| Greedy decoding only | — | Greedy and beam search |

## 1. Setup

In [ ]:
import os, re, math, json, random, collections
import numpy as np
import tensorflow as tf
import keras
from keras import layers, ops
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from nltk.translate.bleu_score import corpus_bleu
import warnings
warnings.filterwarnings('ignore', message='You are using a softmax over axis')  # harmless: first decode step has length 1

print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ---- Paths ----
BASE_DIR      = '/kaggle/input/flickr8k'
IMAGE_DIR     = os.path.join(BASE_DIR, 'Images')
CAPTIONS_FILE = os.path.join(BASE_DIR, 'captions.txt')
WORK_DIR      = '/kaggle/working'

# ---- Data ----
SEED           = 42
IMG_SIZE       = 224          # EfficientNetB0 native resolution -> 7x7 feature grid
N_TRAIN, N_VAL = 6000, 1000   # images; the remaining ~1,091 are the test set
MIN_WORD_COUNT = 5            # rarer training words map to <unk>
MAX_LEN_CAP    = 40           # hard cap on tokens per caption, incl. <start>/<end>

# ---- Model ----
CNN_WEIGHTS = 'imagenet'
D_MODEL, N_HEADS, FF_DIM, N_LAYERS, DROPOUT = 512, 8, 2048, 2, 0.3

# ---- Training ----
BATCH_SIZE, EPOCHS, PEAK_LR, WARMUP_STEPS, WEIGHT_DECAY = 64, 30, 3e-4, 500, 1e-4
BEAM_SIZE = 3

keras.utils.set_random_seed(SEED)
os.makedirs(WORK_DIR, exist_ok=True)

## 2. Load and clean captions

In [ ]:
def clean_caption(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)      # letters only
    return re.sub(r"\s+", " ", text).strip()

captions = collections.defaultdict(list)          # image file -> list of cleaned captions
with open(CAPTIONS_FILE) as f:
    next(f)                                       # header: image,caption
    for line in f:
        line = line.strip()
        if not line:
            continue
        image, caption = line.split(',', 1)       # captions can contain commas
        captions[image].append(clean_caption(caption))

print(f'{len(captions)} images, {sum(map(len, captions.values()))} captions')
print(next(iter(captions.items())))

## 3. Split by image, then build the vocabulary from training captions only

All captions of an image go to the same split, so test images are never seen in training.

In [ ]:
image_ids = sorted(captions)
random.Random(SEED).shuffle(image_ids)
train_ids = image_ids[:N_TRAIN]
val_ids   = image_ids[N_TRAIN:N_TRAIN + N_VAL]
test_ids  = image_ids[N_TRAIN + N_VAL:]
print(f'train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)} images')

PAD, UNK, START, END = '<pad>', '<unk>', '<start>', '<end>'
counts = collections.Counter(w for i in train_ids for c in captions[i] for w in c.split())
itos = [PAD, UNK, START, END] + sorted(w for w, n in counts.items() if n >= MIN_WORD_COUNT)
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)
PAD_ID, UNK_ID, START_ID, END_ID = (stoi[t] for t in (PAD, UNK, START, END))

lengths = [len(c.split()) + 2 for i in train_ids for c in captions[i]]
MAX_LEN = min(MAX_LEN_CAP, int(np.percentile(lengths, 99)))
print(f'vocab {VOCAB_SIZE} (min count {MIN_WORD_COUNT}) | max caption length {MAX_LEN} tokens '
      f'({np.mean(np.array(lengths) > MAX_LEN):.2%} of training captions truncated)')

def encode(caption):
    ids = [START_ID] + [stoi.get(w, UNK_ID) for w in caption.split()][:MAX_LEN - 2] + [END_ID]
    return ids + [PAD_ID] * (MAX_LEN + 1 - len(ids))   # length MAX_LEN+1 -> input/target shifted by one

def decode(ids):
    words = []
    for i in ids:
        if i == END_ID:
            break
        if i not in (PAD_ID, START_ID):
            words.append(itos[i])
    return words

## 4. Extract spatial image features (computed once, then cached)

The CNN's 7×7×1280 output is kept as **49 region vectors** rather than pooled, so the decoder has regions to attend to. Features are stored in float16 (~1 GB for 8,091 images).

In [ ]:
cnn = keras.applications.EfficientNetB0(include_top=False, weights=CNN_WEIGHTS,
                                        input_shape=(IMG_SIZE, IMG_SIZE, 3))
cnn.trainable = False
GRID = cnn.output_shape[1]                 # 7
N_REGIONS, FEAT_DIM = GRID * GRID, cnn.output_shape[-1]

def load_image(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE))   # EfficientNet rescales 0-255 inputs internally

feat_path = os.path.join(WORK_DIR, f'effb0_{IMG_SIZE}_features.npy')
if os.path.exists(feat_path):
    FEATURES = np.load(feat_path)
else:
    paths = [os.path.join(IMAGE_DIR, i) for i in image_ids]
    ds = tf.data.Dataset.from_tensor_slices(paths).map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(64).prefetch(tf.data.AUTOTUNE)
    FEATURES = cnn.predict(ds, verbose=1).reshape(len(paths), N_REGIONS, FEAT_DIM).astype('float16')
    np.save(feat_path, FEATURES)
ROW = {img: r for r, img in enumerate(image_ids)}   # image file -> row in FEATURES
print('features', FEATURES.shape, FEATURES.dtype)

## 5. Data pipeline: one sample per (image, caption) pair

In [ ]:
class CaptionData(keras.utils.PyDataset):
    # Yields ((image_regions, caption_in), caption_out) with caption_out = caption_in shifted by one.
    def __init__(self, ids, batch_size, shuffle, **kwargs):
        super().__init__(**kwargs)
        pairs = [(ROW[i], encode(c)) for i in ids for c in captions[i]]
        self.rows = np.array([r for r, _ in pairs])
        self.seqs = np.array([s for _, s in pairs], dtype='int32')
        self.batch_size, self.shuffle = batch_size, shuffle
        self.order = np.arange(len(pairs))
        self.rng = np.random.default_rng(SEED)
        if shuffle:
            self.rng.shuffle(self.order)

    def __len__(self):
        return math.ceil(len(self.order) / self.batch_size)

    def __getitem__(self, idx):
        b = self.order[idx * self.batch_size:(idx + 1) * self.batch_size]
        seq = self.seqs[b]
        return (FEATURES[self.rows[b]].astype('float32'), seq[:, :-1]), seq[:, 1:]

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.order)

train_data = CaptionData(train_ids, BATCH_SIZE, shuffle=True)
val_data   = CaptionData(val_ids, BATCH_SIZE, shuffle=False)
print(f'{len(train_data.order)} training pairs, {len(train_data)} steps/epoch')

## 6. Model

For every caption position *t*, cross-attention computes weights over the 49 image regions, $\alpha_{t,r} = \mathrm{softmax}_r(q_t \cdot k_r / \sqrt{d})$. So each word gets its own image context. In the old block, every position received the same context.

In [ ]:
class ImageEncoder(layers.Layer):
    # Projects the 49 CNN region vectors to D_MODEL and adds learned 2D position embeddings.
    def __init__(self, d_model, n_regions, dropout, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Dense(d_model, activation='gelu')
        self.pos = layers.Embedding(n_regions, d_model)
        self.norm = layers.LayerNormalization()
        self.drop = layers.Dropout(dropout)

    def call(self, regions, training=False):
        x = self.proj(regions) + self.pos(ops.arange(ops.shape(regions)[1]))
        return self.drop(self.norm(x), training=training)


class DecoderBlock(layers.Layer):
    def __init__(self, d_model, n_heads, ff_dim, dropout, **kwargs):
        super().__init__(**kwargs)
        self.self_attn = layers.MultiHeadAttention(n_heads, d_model // n_heads, dropout=dropout)
        self.cross_attn = layers.MultiHeadAttention(n_heads, d_model // n_heads, dropout=dropout)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation='gelu'), layers.Dropout(dropout),
                                     layers.Dense(d_model)])
        self.norm1, self.norm2, self.norm3 = (layers.LayerNormalization() for _ in range(3))
        self.drop = layers.Dropout(dropout)

    def call(self, x, image, training=False):
        # Causal mask: position t only sees tokens <= t. Captions are post-padded, so real tokens never see padding.
        h = self.self_attn(x, x, use_causal_mask=True, training=training)
        x = self.norm1(x + self.drop(h, training=training))
        h, attn = self.cross_attn(x, image, return_attention_scores=True, training=training)
        x = self.norm2(x + self.drop(h, training=training))
        x = self.norm3(x + self.drop(self.ffn(x, training=training), training=training))
        return x, attn                                   # attn: (batch, heads, T, regions)


class Captioner(keras.Model):
    def __init__(self, vocab_size, max_len, n_regions, d_model, n_heads, ff_dim, n_layers, dropout, **kwargs):
        super().__init__(**kwargs)
        self.encoder = ImageEncoder(d_model, n_regions, dropout)
        self.tok_emb = layers.Embedding(vocab_size, d_model)
        self.pos_emb = layers.Embedding(max_len, d_model)
        self.emb_drop = layers.Dropout(dropout)
        self.blocks = [DecoderBlock(d_model, n_heads, ff_dim, dropout) for _ in range(n_layers)]
        self.head = layers.Dense(vocab_size)             # logits
        self.d_model = d_model

    def decode_with_attention(self, inputs, training=False):
        regions, tokens = inputs
        image = self.encoder(regions, training=training)
        x = self.tok_emb(tokens) * math.sqrt(self.d_model) + self.pos_emb(ops.arange(ops.shape(tokens)[1]))
        x = self.emb_drop(x, training=training)
        attns = []
        for block in self.blocks:
            x, a = block(x, image, training=training)
            attns.append(a)
        return self.head(x), attns

    def call(self, inputs, training=False):
        return self.decode_with_attention(inputs, training=training)[0]


def masked_loss(y_true, logits):
    # Token-normalised cross-entropy: mean over real tokens in the batch, padding excluded.
    loss = keras.losses.sparse_categorical_crossentropy(y_true, logits, from_logits=True)
    mask = ops.cast(ops.not_equal(y_true, PAD_ID), loss.dtype)
    return ops.sum(loss * mask) / ops.maximum(ops.sum(mask), 1.0)

def masked_accuracy(y_true, logits):
    y_true = ops.cast(y_true, 'int32')
    hit = ops.cast(ops.equal(y_true, ops.cast(ops.argmax(logits, axis=-1), 'int32')), 'float32')
    mask = ops.cast(ops.not_equal(y_true, PAD_ID), 'float32')
    return ops.sum(hit * mask) / ops.maximum(ops.sum(mask), 1.0)

model = Captioner(VOCAB_SIZE, MAX_LEN, N_REGIONS, D_MODEL, N_HEADS, FF_DIM, N_LAYERS, DROPOUT)
model((np.zeros((1, N_REGIONS, FEAT_DIM), 'float32'), np.zeros((1, MAX_LEN), 'int32')))   # build
schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0, warmup_target=PEAK_LR, warmup_steps=WARMUP_STEPS,
    decay_steps=max(1, EPOCHS * len(train_data) - WARMUP_STEPS), alpha=0.05)
model.compile(optimizer=keras.optimizers.AdamW(schedule, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
              loss=masked_loss, metrics=[masked_accuracy])
model.summary()

## 7. Train (early stopping on validation loss)

In [ ]:
ckpt_path = os.path.join(WORK_DIR, 'captioner.weights.h5')
history = model.fit(
    train_data, validation_data=val_data, epochs=EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
               keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True,
                                               save_weights_only=True)])

with open(os.path.join(WORK_DIR, 'vocab.json'), 'w') as f:
    json.dump({'itos': itos, 'max_len': MAX_LEN}, f)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, a in zip(['loss', 'masked_accuracy'], ax):
    a.plot(history.history[k], label='train'); a.plot(history.history['val_' + k], label='val')
    a.set_title(k); a.set_xlabel('epoch'); a.legend()
plt.show()

## 8. Decoding: greedy and beam search

In [ ]:
def greedy_decode(regions):
    # Batched greedy decoding. regions: (B, 49, FEAT_DIM) -> list of word lists.
    B = len(regions)
    tokens = np.full((B, 1), START_ID, dtype='int32')
    done = np.zeros(B, dtype=bool)
    for _ in range(MAX_LEN - 1):
        logits = model((regions, tokens), training=False)
        nxt = ops.convert_to_numpy(ops.argmax(logits[:, -1], axis=-1)).astype('int32')
        nxt[done] = PAD_ID
        tokens = np.concatenate([tokens, nxt[:, None]], axis=1)
        done |= nxt == END_ID
        if done.all():
            break
    return [decode(t[1:]) for t in tokens]


def beam_decode(region, beam_size=BEAM_SIZE, alpha=0.7):
    # Beam search for one image. Score = log-prob / length**alpha (length normalisation).
    beams = [([START_ID], 0.0)]
    finished = []
    for _ in range(MAX_LEN - 1):
        seqs = np.array([s for s, _ in beams], dtype='int32')
        logits = model((np.repeat(region[None], len(beams), 0), seqs), training=False)
        logp = ops.convert_to_numpy(ops.log_softmax(logits[:, -1], axis=-1))
        cands = []
        for (seq, score), lp in zip(beams, logp):
            for tok in np.argsort(lp)[-beam_size:]:
                cands.append((seq + [int(tok)], score + float(lp[tok])))
        cands.sort(key=lambda c: c[1], reverse=True)
        beams = []
        for seq, score in cands:
            (finished if seq[-1] == END_ID else beams).append((seq, score))
            if len(beams) == beam_size:
                break
        if len(finished) >= beam_size or not beams:
            break
    pool = finished or beams
    best = max(pool, key=lambda c: c[1] / (len(c[0]) - 1) ** alpha)
    return decode(best[0][1:])

## 9. Evaluate on held-out test images

Corpus BLEU against all 5 references per image, with special tokens removed from both sides.

In [ ]:
references = [[c.split() for c in captions[i]] for i in test_ids]
test_regions = FEATURES[[ROW[i] for i in test_ids]].astype('float32')

greedy = []
for s in tqdm(range(0, len(test_ids), 128), desc='greedy'):
    greedy += greedy_decode(test_regions[s:s + 128])
beam = [beam_decode(r) for r in tqdm(test_regions, desc=f'beam={BEAM_SIZE}')]

def bleu_scores(hyps):
    w = [(1, 0, 0, 0), (.5, .5, 0, 0), (1/3, 1/3, 1/3, 0), (.25, .25, .25, .25)]
    return [round(100 * corpus_bleu(references, hyps, weights=wi), 1) for wi in w]

results = {'greedy': bleu_scores(greedy), f'beam (k={BEAM_SIZE})': bleu_scores(beam)}
print(f"{'decoding':<14}" + ''.join(f'BLEU-{n:<4}' for n in range(1, 5)))
for k, v in results.items():
    print(f'{k:<14}' + ''.join(f'{x:<9}' for x in v))
with open(os.path.join(WORK_DIR, 'test_bleu.json'), 'w') as f:
    json.dump({'n_test_images': len(test_ids), **results}, f, indent=2)

## 10. Sample captions on test images

In [ ]:
def show_examples(ids, n=6):
    fig, axes = plt.subplots(math.ceil(n / 3), 3, figsize=(18, 6 * math.ceil(n / 3)))
    for ax, i in zip(axes.flat, random.Random(SEED).sample(ids, n)):
        ax.imshow(Image.open(os.path.join(IMAGE_DIR, i)))
        pred = ' '.join(beam_decode(FEATURES[ROW[i]].astype('float32')))
        ax.set_title(f'pred: {pred}\nref: {captions[i][0]}', fontsize=10, loc='left')
        ax.axis('off')
    plt.tight_layout(); plt.show()

show_examples(test_ids)

## 11. Where does the model look? Per-word attention maps

Last decoder layer, cross-attention averaged over heads. Each 7×7 map is upsampled onto the image.

In [ ]:
def plot_attention(image_id, layer=-1):
    region = FEATURES[ROW[image_id]].astype('float32')
    words = beam_decode(region)
    tokens = np.array([[START_ID] + [stoi.get(w, UNK_ID) for w in words]], dtype='int32')
    _, attns = model.decode_with_attention((region[None], tokens), training=False)
    attn = ops.convert_to_numpy(attns[layer])[0].mean(axis=0)        # (T, regions)
    img = np.array(Image.open(os.path.join(IMAGE_DIR, image_id)).resize((IMG_SIZE, IMG_SIZE)))
    cols = 5
    rows = math.ceil(len(words) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for t, (ax, w) in enumerate(zip(axes.flat, words)):
        heat = np.array(Image.fromarray(attn[t].reshape(GRID, GRID).astype('float32')).resize(
            (IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        ax.imshow(img); ax.imshow(heat, cmap='jet', alpha=0.45); ax.set_title(w)
    plt.suptitle(' '.join(words)); plt.tight_layout(); plt.show()

for i in random.Random(SEED + 1).sample(test_ids, 2):
    plot_attention(i)

## 12. Caption any image (file path or URL)

In [ ]:
def caption_image(path_or_url):
    if path_or_url.startswith('http'):
        path_or_url = keras.utils.get_file(origin=path_or_url)
    region = cnn.predict(load_image(path_or_url)[None], verbose=0).reshape(1, N_REGIONS, FEAT_DIM)[0]
    words = beam_decode(region)
    plt.imshow(Image.open(path_or_url)); plt.axis('off'); plt.title(' '.join(words)); plt.show()
    return ' '.join(words)

caption_image(os.path.join(IMAGE_DIR, test_ids[0]))
# caption_image('https://<any .jpg url>')

## Limitations and next steps

- **The CNN is frozen.** Fine-tuning the top EfficientNet blocks at a low learning rate usually adds a few BLEU-4 points.
- **Flickr8k is small** (6k training images). The same code scales to Flickr30k or MS-COCO.
- **BLEU only.** CIDEr and METEOR (via `pycocoevalcap`) are the standard captioning metrics and correlate better with human judgement.
- **Teacher forcing leaves an exposure-bias gap** between training and decoding. Scheduled sampling or self-critical (CIDEr-optimised) training addresses it.